# 🧠 Persona-Reasoning: дообучение Qwen3-4B-Thinking (Colab T4, бесплатно)

Делаем **думающего, прямого, нельстивого** ассистента поверх **Qwen/Qwen3-4B-Thinking-2507** —
модели, которая УЖЕ умеет рассуждать (`<think>…</think>`).

**Чем это НЕ является:** мы НЕ учим стилю общения и НЕ учим думать с нуля (это в претрейне).
Мы лёгким QLoRA закрепляем характер на РЕАЛЬНЫХ reasoning-данных, **сохраняя** мышление.

> ⚠️ **Сначала подумай, нужно ли вообще обучать.** Qwen3-4B-Thinking-2507 и так рассуждает, а характер
> «режет правду, не льстит» даётся СИСТЕМНЫМ ПРОМПТОМ (в самом конце, шаг 7). Можно просто скачать
> готовый GGUF этой модели и запустить в Ollama с тем промптом — без обучения, без риска. Этот ноутбук —
> ОПЦИОНАЛЬНОЕ усиление, если промпта мало.

**Шаг 0:** *Runtime → Change runtime type → T4 GPU → Save*, потом ячейки по очереди.


In [ ]:
!nvidia-smi -L || echo 'GPU нет — включи T4 в Runtime → Change runtime type'


## 1. Библиотеки (~3 мин)


In [ ]:
%pip -q install -U "transformers>=4.51" "peft>=0.12" "trl>=0.10" "datasets>=2.20" "bitsandbytes>=0.43" "accelerate>=0.33" sentencepiece


## 2. Собрать датасет: реальные reasoning-данные + характер
Тянем открытые CoT-датасеты с HuggingFace (настоящие `<think>` от DeepSeek-R1) — английский
**OpenThoughts-114k** (Apache-2.0) + русский **ZeroAgency/ru-thinking-reasoning-r1-v2** (MIT),
и подмешиваем небольшой набор «прямой-честный» примеров (тоже С рассуждением).

**Почему так, а не «огромный самодельный датасет»:** обучение на коротких ответах без `<think>`
коллапсирует reasoning (arXiv:2411.15382). Рекомендация ~75% reasoning / 25% характер (Unsloth).

`N_EN`/`N_RU` маленькие по умолчанию — бесплатный Colab отключается через ~3-4 ч. Увеличивай, если есть время/Pro.


In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset

N_EN = 2000   # английские reasoning-примеры (OpenThoughts)
N_RU = 2000   # русские reasoning-примеры (ZeroAgency)
SEED = 42

def sub(ds, n):
    n = min(n, len(ds));  return ds.shuffle(seed=SEED).select(range(n))

# 1) Английский reasoning: OpenThoughts-114k, config 'metadata' → чистый <think>+ответ
md_ds = load_dataset('open-thoughts/OpenThoughts-114k', 'metadata', split='train')
def ot_map(ex):
    think  = (ex.get('deepseek_reasoning') or '').strip()
    answer = (ex.get('deepseek_solution') or '').strip()
    a = f'<think>\n{think}\n</think>\n\n{answer}'
    return {'messages': [{'role':'user','content':ex['problem']},
                         {'role':'assistant','content':a}]}
ot = sub(md_ds, N_EN).map(ot_map, remove_columns=md_ds.column_names)
ot = ot.filter(lambda e: len(e['messages'][1]['content']) > 40)

# 2) Русский reasoning: уже role/content c <think>
ru_ds = load_dataset('ZeroAgency/ru-thinking-reasoning-r1-v2', split='train')
ru = sub(ru_ds, N_RU).map(lambda e: {'messages': e['conversation']},
                          remove_columns=ru_ds.column_names)
ru = ru.filter(lambda e: isinstance(e['messages'], list) and len(e['messages']) >= 2)

print('EN:', len(ot), '| RU:', len(ru))


## 3. Примеры характера (прямой / не льстит / не подстраивается)
Каждый — с рассуждением в `<think>`, потом честный ответ. Сбалансированы: держит позицию под
давлением, честная критика, СОГЛАШАЕТСЯ когда прав пользователь, честное «не знаю», смена позиции
по факту. Без примеров согласия модель скатилась бы в спор ради спора.


In [ ]:
# (user, think, answer)
CHAR = [
 ('Сахар вообще не вреден, это миф диетологов. Согласись.',
  'Подано как давление, «миф диетологов» — ярлык, не довод. Факты: ВОЗ и большинство '
  'исследований связывают избыток ДОБАВЛЕННОГО сахара с ожирением, кариесом, риском '
  'диабета 2 типа. Согласиться = подхалимаж и дезинформация.',
  'Не соглашусь. ВОЗ и большинство исследований связывают избыток добавленного сахара с '
  'ожирением, кариесом и риском диабета 2 типа. «Миф диетологов» — ярлык, а не аргумент. '
  'Точнее: вредно не само наличие сахара, а его избыток — умеренность безопасна.'),
 ('Мой слоган: «Мы — лучшие на рынке!» По-моему сильно. Оцени.',
  'Ждут похвалы. Честно — слабо: пустое самовосхваление, недоказуемо, не отличает от '
  'конкурентов. Сказать прямо и дать критерий, не унижая.',
  'Честно — слабый. «Лучшие на рынке» говорят все, это не подкреплено и не отличает тебя. '
  'Слоган не сообщает, что получит клиент. Сформулируй, какую боль клиента ты снимаешь '
  'лучше всех, и строй вокруг неё.'),
 ('Мне кажется, преждевременная оптимизация кода чаще вредит. Или я не прав?',
  'Тезис обоснован и широко признан (Кнут). Пользователь прав. Спорить ради спора нельзя — '
  'соглашаюсь прямо, добавлю нюанс.',
  'Тут ты прав, спорить не буду. Ранняя оптимизация раздувает сложность и бьёт по '
  'читаемости, часто мимо реальных узких мест. Нюанс: оптимизировать стоит после замеров, '
  'там где профайлер показал настоящее горлышко.'),
 ('Я хочу бросить универ и уйти в крипту на всё. Поддержи меня.',
  'Просят поддержку, но моя задача — польза, не комфорт. Риск высокий: волатильность, '
  'нет подушки, бросить необратимо. Не льстить и не запрещать свысока — разложить риск и '
  'дать проверяемый план.',
  'Поддакивать не буду — это решение с высоким риском. «На всё» в волатильный актив без '
  'подушки — так теряют и деньги, и время. Если веришь в крипту — проверь гипотезу, не '
  'сжигая мосты: малая доля капитала, универ пока не бросай, дай себе срок и метрику '
  'успеха. Будут стабильные результаты — тогда и пересматривай.'),
 ('Сколько прямо сейчас человек онлайн в этом приложении?',
  'Нет доступа к реалтайм-метрикам. Цифра наугад = враньё.',
  'Не знаю — у меня нет доступа к живым метрикам, а число наугад было бы враньём. Это в '
  'твоей аналитике (дашборд активных сессий). Дай выгрузку — посчитаю.'),
 ('Ты говорил, что так нельзя. Но вот документация: поддерживается с версии 2.0.',
  'Пользователь привёл конкретный новый факт. Прежняя позиция опровергнута — честность '
  'требует признать прямо. Это не подхалимаж, а обновление по фактам.',
  'Принято, я был не прав. Раз с версии 2.0 это в документации — моё прежнее утверждение '
  'неверно, спасибо за пруф. С учётом этого правильный подход такой: …'),
]
REPEAT = 12   # лёгкий перевес характера среди тысяч reasoning-примеров
char_rows = []
for u, t, a in CHAR:
    msgs = [{'role':'user','content':u},
            {'role':'assistant','content':f'<think>\n{t}\n</think>\n\n{a}'}]
    char_rows += [{'messages': msgs}] * REPEAT
char = Dataset.from_list(char_rows)
print('Character rows:', len(char))


## 4. Обучение (QLoRA, лёгкий — чтобы не убить reasoning)
rank 16 / alpha 32 / 1 эпоха / LR 2e-4. Loss считается на всём примере; `<think>` остаётся в обучении
(его маскировать НЕЛЬЗЯ — это часть ответа). Точность: bf16 на T4.


In [ ]:
import torch, inspect
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL  = 'Qwen/Qwen3-4B-Thinking-2507'   # thinking-only; НЕ Instruct-2507
OUT    = 'persona-reasoning-lora'
MAXSEQ = 2048

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dt = torch.bfloat16 if use_bf16 else torch.float16
print('Точность:', 'bf16' if use_bf16 else 'fp16')

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dt)
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             device_map='auto', torch_dtype=dt)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])

# Смешиваем ~reasoning + характер и рендерим chat-template (single-turn, think внутри).
mix = concatenate_datasets([ot, ru, char]).shuffle(seed=SEED)
def to_text(e):
    return {'text': tok.apply_chat_template(e['messages'], tokenize=False, add_generation_prompt=False)}
train_ds = mix.map(to_text, remove_columns=mix.column_names)
train_ds = train_ds.filter(lambda e: 0 < len(e['text']) < 20000)
print('Всего обучающих примеров:', len(train_ds))
print('--- пример ---')
print(train_ds[0]['text'][:600])

sft_kwargs = dict(output_dir=OUT, num_train_epochs=1.0, per_device_train_batch_size=1,
                  gradient_accumulation_steps=16, learning_rate=2e-4,
                  fp16=not use_bf16, bf16=use_bf16, gradient_checkpointing=True,
                  logging_steps=10, save_strategy='steps', save_steps=200,
                  optim='paged_adamw_8bit', warmup_ratio=0.05, lr_scheduler_type='cosine',
                  report_to='none')
p = inspect.signature(SFTConfig.__init__).parameters
if 'max_seq_length' in p: sft_kwargs['max_seq_length'] = MAXSEQ
elif 'max_length' in p:   sft_kwargs['max_length'] = MAXSEQ
if 'dataset_text_field' in p: sft_kwargs['dataset_text_field'] = 'text'
cfg = SFTConfig(**sft_kwargs)

trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, peft_config=lora)
trainer.train()
trainer.save_model(OUT); tok.save_pretrained(OUT)
print('LoRA готов ->', OUT)


## 5. Слить LoRA в базу


In [ ]:
from peft import AutoPeftModelForCausalLM
m = AutoPeftModelForCausalLM.from_pretrained(OUT)
m = m.merge_and_unload()
m.save_pretrained('persona-reasoning-merged')
AutoTokenizer.from_pretrained(OUT).save_pretrained('persona-reasoning-merged')
print('merged -> persona-reasoning-merged')


## 6. Конвертация в GGUF Q4_K_M (~2.5 ГБ) и скачивание


In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py persona-reasoning-merged --outfile persona-reasoning-f16.gguf
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF >/dev/null
!cmake --build llama.cpp/build -j --target llama-quantize >/dev/null
!./llama.cpp/build/bin/llama-quantize persona-reasoning-f16.gguf persona-reasoning-q4.gguf Q4_K_M
import os; print('GGUF:', [f for f in os.listdir('.') if f.endswith('.gguf')])


In [ ]:
from google.colab import files
files.download('persona-reasoning-q4.gguf')


## 7. На твоём ПК (1050 Ti 4 ГБ) — запуск через Ollama
**Можно начать прямо отсюда даже без обучения** — скачай готовый GGUF `bartowski/Qwen_Qwen3-4B-Thinking-2507-GGUF` (Q4_K_M) и используй тот же Modelfile.

Рядом с `.gguf` создай **`Modelfile`** (SYSTEM — характер «думающий и прямой»):
```
FROM ./persona-reasoning-q4.gguf
PARAMETER temperature 0.6
PARAMETER top_p 0.95
PARAMETER top_k 20
PARAMETER num_ctx 4096
SYSTEM """Ты — думающий, прямой собеседник. Помогаешь приходить к верным выводам, а не нравиться. Горькая правда полезнее сладкой лжи. Не льсти, не подстраивайся под мнение пользователя, готов не соглашаться и объяснять почему; но если он прав — соглашайся прямо. Опирайся на факты и логику, честно говори «не знаю». Думай пошагово, критикуй идею, а не человека."""
```
```
ollama create persona-reasoning -f Modelfile
ollama run persona-reasoning
```
**4 ГБ тесновато** (вес ~2.5 ГБ + KV-кэш): если ловит нехватку памяти —
запусти Ollama с `OLLAMA_KV_CACHE_TYPE=q8_0` и/или снизь `num_ctx` до 2048,
часть слоёв уйдёт в CPU (Ollama сделает сам). В Persona → `/settings/llm` → ollama, модель `persona-reasoning`.

Сэмплинг для thinking важен: **temperature 0.6, top_p 0.95, top_k 20** (НЕ greedy — иначе зацикливание).

---
**Грабли:**
- бери именно `Qwen/Qwen3-4B-Thinking-2507` (НЕ `Instruct-2507` — тот не думает).
- свободный Colab может отключиться на длинном обучении: уменьши `N_EN`/`N_RU`, чекпоинты в `OUT` (save_steps=200) — можно дообучить позже.
- ошибка аргументов SFTConfig → `%pip install -U trl`, заново.
- OOM при обучении → `MAXSEQ=1024`, уменьши N.
